[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-03-conditional-edges.ipynb#scrollTo=aa112233)

---
# Day 3 · Conditional Edges and Routing Logic
**certified-journeys / ai-agents-certified** · Day 3 · Graph Control Flow

> **Goal for today:** Implement a keyword-based router that inspects message content and routes inputs to one of three specialised nodes (math, code, or clarification), wire conditional edges correctly, add a fallback branch, and write unit tests that drive the graph through every branch using mock responses.

In [ ]:
%pip install -q langgraph langchain-openai langchain-core

## Step 1 · How Conditional Edges Work

A **conditional edge** defers the routing decision to a Python function called at runtime. Instead of a fixed `A → B` link, you write a **routing function** that inspects state and returns a node name string.

```python
graph.add_conditional_edges(
    source_node,          # which node's output triggers the decision
    routing_function,     # (state) -> str — returns a node name or END
    path_map              # optional {"returned_str": "actual_node_name"} alias map
)
```

**Critical contract:** the routing function must return a string that **exactly matches** a registered node name or the `END` constant. A typo causes a `ValueError` at compile time (in newer LangGraph) or a silent hang at runtime.

| Edge type | API | Use when |
|-----------|-----|----------|
| Fixed | `add_edge(A, B)` | Always go to B after A |
| Conditional | `add_conditional_edges(A, fn)` | Branch based on runtime state |
| Conditional + alias | `add_conditional_edges(A, fn, {"label": "node"})` | Router returns friendly labels, graph maps to node names |

In [ ]:
# Imports for the full routing graph
from typing import TypedDict, Annotated, Literal
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

# State schema for the routing graph
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]   # full conversation history
    intent: str                               # set by the router; read by worker nodes
    result: str                               # set by the worker node that handles the request

### What just happened?

- **`intent`** is a scalar string — no reducer needed; the router node will set it once.
- **`result`** is also scalar — the worker node that handles the request writes it.
- **`messages`** uses `add_messages` so conversation history accumulates across turns.
- The routing logic will read `messages[-1].content` — the most recent human message — to decide the intent.

## Step 2 · The Routing Function — Keyword Intent Classification

The routing function is **just a Python function**. It receives the current state and returns a string. LangGraph calls it **after the source node** has updated state.

Design decisions:
- Check keywords in priority order (most specific first)
- Always have a **fallback** return value — never let the function fall off the end with no return
- Keep routing logic **pure** — no side effects, no LLM calls inside the router itself

In [ ]:
# The routing node: classifies intent and writes it to state
def classify_intent(state: RouterState) -> dict:
    """
    Reads the last message and sets state['intent'].
    This node runs first; the conditional edge reads intent from state.
    """
    last_message = state["messages"][-1]
    content = last_message.content.lower()

    # Priority-ordered keyword matching
    math_keywords   = ["calculate", "math", "equation", "sum", "multiply", "divide", "+", "-", "*", "/"]
    code_keywords   = ["code", "python", "function", "script", "debug", "program", "implement"]

    if any(kw in content for kw in math_keywords):
        return {"intent": "math"}
    elif any(kw in content for kw in code_keywords):
        return {"intent": "code"}
    else:
        return {"intent": "unclear"}


# The routing FUNCTION (called by add_conditional_edges — not a node)
# Reads state and returns the NAME of the next node to run
def route_by_intent(state: RouterState) -> Literal["math_handler", "code_handler", "clarify_handler"]:
    """
    Pure routing function — no side effects.
    Returns one of the three registered node names.
    """
    intent = state.get("intent", "unclear")

    if intent == "math":
        return "math_handler"
    elif intent == "code":
        return "code_handler"
    else:
        # Fallback: unknown intent → clarification node (never crashes the graph)
        return "clarify_handler"


# Quick sanity check — routing function is testable without a graph
test_state_math    = {"messages": [HumanMessage(content="calculate 5 * 7")], "intent": "math", "result": ""}
test_state_code    = {"messages": [HumanMessage(content="write a python function")], "intent": "code", "result": ""}
test_state_unclear = {"messages": [HumanMessage(content="tell me a story")], "intent": "unclear", "result": ""}

print("Math    →", route_by_intent(test_state_math))
print("Code    →", route_by_intent(test_state_code))
print("Unclear →", route_by_intent(test_state_unclear))

### What just happened?

- **`classify_intent`** is a regular node — it writes to state (`intent` field) and returns a dict.
- **`route_by_intent`** is the routing function — it reads from state and returns a node name string. It is NOT added as a node; it's passed to `add_conditional_edges`.
- **Separating classification (node) from routing (function)** is the cleanest pattern: state holds the decision so you can inspect it, log it, and test it independently.
- **`Literal[...]` type hint** documents the valid return values and enables IDE checking — it's not enforced at runtime but makes the contract clear.

## Step 3 · The Worker Nodes

Each worker node handles one branch. In a real agent these would call specialised LLMs or tools. Here we use **mock responses** so the notebook runs without API keys.

In [ ]:
def math_handler(state: RouterState) -> dict:
    """
    Handles math questions.
    Production: call a math-focused LLM or a calculation tool.
    Mock: echo the question with a canned math response.
    """
    question = state["messages"][-1].content
    # Production equivalent: response = math_llm.invoke(state["messages"])
    answer = f"[MATH] I'll solve: '{question}'. The answer is computed step by step."
    return {
        "result": answer,
        "messages": [AIMessage(content=answer)]
    }


def code_handler(state: RouterState) -> dict:
    """
    Handles programming questions.
    Production: call a code-optimised LLM (e.g., gpt-4o with system prompt for code).
    Mock: return a structured code-style response.
    """
    question = state["messages"][-1].content
    answer = (
        f"[CODE] Here's a Python solution for: '{question}'.\n"
        "```python\ndef solution():\n    pass  # implementation here\n```"
    )
    return {
        "result": answer,
        "messages": [AIMessage(content=answer)]
    }


def clarify_handler(state: RouterState) -> dict:
    """
    Fallback: intent unclear — ask the user to rephrase.
    This node prevents the graph from crashing on unknown inputs.
    """
    answer = (
        "I'm not sure what you need. Could you rephrase? "
        "Try starting with 'calculate', 'code', or 'explain'."
    )
    return {
        "result": answer,
        "messages": [AIMessage(content=answer)]
    }


print("Worker node functions defined.")

### What just happened?

- Each worker node is independent — it doesn't know about other branches. This is intentional: worker nodes are swappable without touching the routing logic.
- All three workers write to both `result` (a plain string summary) and `messages` (the conversation history). The `messages` field uses `add_messages`, so the AI reply is appended rather than replacing the human message.
- **The fallback (`clarify_handler`) is the most important node** — it guarantees the graph always reaches END gracefully, even for unexpected inputs.

## Step 4 · Wiring the Conditional Edge Graph

In [ ]:
# Build the full conditional routing graph
router_builder = StateGraph(RouterState)

# --- Register nodes ---
router_builder.add_node("classify",       classify_intent)  # classification node
router_builder.add_node("math_handler",   math_handler)
router_builder.add_node("code_handler",   code_handler)
router_builder.add_node("clarify_handler",clarify_handler)

# --- Fixed entry edge ---
router_builder.add_edge(START, "classify")

# --- Conditional edge: after classify, call route_by_intent to decide next node ---
# The third argument (path_map) is optional here — route_by_intent already returns
# exact node names, so no alias mapping is needed.
router_builder.add_conditional_edges(
    "classify",
    route_by_intent,
    # path_map (optional) — shown here for documentation:
    {
        "math_handler":    "math_handler",
        "code_handler":    "code_handler",
        "clarify_handler": "clarify_handler",
    }
)

# --- All worker nodes terminate at END ---
router_builder.add_edge("math_handler",    END)
router_builder.add_edge("code_handler",    END)
router_builder.add_edge("clarify_handler", END)

router_graph = router_builder.compile()

# Visualise the graph structure
print(router_graph.get_graph().draw_mermaid())

### What just happened?

- **`add_conditional_edges(source, fn, path_map)`** — LangGraph calls `fn(state)` after `source` runs. The return value is looked up in `path_map` to get the actual node name.
- When `path_map` values match the keys (as here), the map is redundant but makes the valid branches **explicit and inspectable** — always include it for readability.
- The Mermaid output shows a diamond (decision node) at `classify` with three outgoing branches — that's the visual signature of conditional edges.
- **Every branch must end at a registered node or `END`** — an orphan branch causes a compile error.

## Step 5 · Running the Graph Through All Three Branches

In [ ]:
def run_router(user_input: str) -> dict:
    """Helper: invoke the router graph with a single user message."""
    initial_state = {
        "messages": [HumanMessage(content=user_input)],
        "intent": "",
        "result": "",
    }
    return router_graph.invoke(initial_state)


# Test each branch
test_inputs = [
    ("calculate 12 * 8 + 5",         "math branch"),
    ("write a python function to sort a list", "code branch"),
    ("tell me about the weather today", "fallback branch"),
]

for user_input, expected_branch in test_inputs:
    result = run_router(user_input)
    print(f"Input   : {user_input!r}")
    print(f"Expected: {expected_branch}")
    print(f"Intent  : {result['intent']}")
    print(f"Result  : {result['result'][:80]}..." if len(result['result']) > 80 else f"Result  : {result['result']}")
    print(f"Messages: {len(result['messages'])} total")
    print()

### What just happened?

- **All three branches ran correctly** — the math, code, and clarify handlers each produced a distinct `result` and appended an `AIMessage` to the conversation.
- The `intent` field in the final state proves that `classify_intent` ran before routing: the routing function read the intent that the node wrote.
- **`messages` has 2 entries** after each run: the original `HumanMessage` plus the `AIMessage` from the worker. The `add_messages` reducer accumulated them correctly.

## Step 6 · Streaming to See Which Branch Was Taken

When debugging routing issues, `stream` is more useful than `invoke` — it shows which node ran at each step, so you can verify the edge was taken.

In [ ]:
print("=== Stream trace for 'calculate 7 + 3' ===")
for step in router_graph.stream({
    "messages": [HumanMessage(content="calculate 7 + 3")],
    "intent": "",
    "result": ""
}):
    node_name = next(iter(step.keys()))
    update    = step[node_name]
    print(f"  Node ran: '{node_name}'")
    if "intent" in update:
        print(f"    → set intent = {update['intent']!r}")
    if "result" in update and update["result"]:
        print(f"    → set result = {update['result'][:60]!r}...")
print("=== Graph reached END ===\n")

### What just happened?

- Stream output confirms the execution order: `classify` → `math_handler` → END. The conditional edge is invisible in the stream (it's not a node), but the sequence of nodes proves which branch was taken.
- **For debugging routing bugs** — if the wrong handler runs, check: (1) what `classify_intent` wrote to `intent`, (2) what `route_by_intent` returns for that intent value, (3) that the return value exactly matches the registered node name.

## Step 7 · Unit Testing — Drive All Branches with Mock LLM Responses

Unit testing a LangGraph graph means:
1. Providing deterministic input (no real LLM needed)
2. Asserting on the **final state dict** — not on implementation internals
3. Testing every branch, not just the happy path

We use Python's built-in `unittest` so this runs in any environment including Colab.

In [ ]:
import unittest
from langchain_core.messages import HumanMessage

def invoke_graph(user_text: str) -> dict:
    """Test helper: wraps the router graph for clean assertions."""
    return router_graph.invoke({
        "messages": [HumanMessage(content=user_text)],
        "intent": "",
        "result": "",
    })


class TestRouterGraph(unittest.TestCase):

    def test_math_branch_keyword_calculate(self):
        """'calculate' keyword → math_handler → intent=='math'"""
        result = invoke_graph("calculate 100 / 4")
        self.assertEqual(result["intent"], "math")
        self.assertIn("[MATH]", result["result"])

    def test_math_branch_keyword_equation(self):
        """'equation' keyword also routes to math."""
        result = invoke_graph("solve this equation: 2x + 5 = 11")
        self.assertEqual(result["intent"], "math")

    def test_code_branch_keyword_python(self):
        """'python' keyword → code_handler → intent=='code'"""
        result = invoke_graph("write a python function to reverse a string")
        self.assertEqual(result["intent"], "code")
        self.assertIn("[CODE]", result["result"])

    def test_code_branch_keyword_debug(self):
        """'debug' keyword routes to code handler."""
        result = invoke_graph("debug my script — it crashes on line 3")
        self.assertEqual(result["intent"], "code")

    def test_fallback_branch_unknown_intent(self):
        """Unrecognised input → clarify_handler → intent=='unclear'"""
        result = invoke_graph("what is the capital of France?")
        self.assertEqual(result["intent"], "unclear")
        self.assertIn("rephrase", result["result"].lower())

    def test_fallback_branch_empty_input(self):
        """Empty string input → fallback (graph never crashes)."""
        result = invoke_graph("")
        self.assertEqual(result["intent"], "unclear")

    def test_messages_accumulated(self):
        """Final state always has 2 messages: human + AI reply."""
        for text in ["calculate 1+1", "write python code", "random text here"]:
            result = invoke_graph(text)
            self.assertEqual(len(result["messages"]), 2,
                msg=f"Expected 2 messages for input: {text!r}")

    def test_math_priority_over_code(self):
        """When both 'calculate' and 'python' appear, math wins (checked first)."""
        result = invoke_graph("calculate the output of my python script")
        self.assertEqual(result["intent"], "math",
            msg="Math keywords should take priority over code keywords")


# Run in Colab / script (not inside pytest)
suite  = unittest.TestLoader().loadTestsFromTestCase(TestRouterGraph)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print()
print(f"Tests passed: {result.testsRun - len(result.failures) - len(result.errors)}/{result.testsRun}")

### What just happened?

- **8 test cases cover all three branches** including edge cases: empty input, priority conflict (math vs code keywords), and message count verification.
- Tests assert on **final state fields** (`intent`, `result`, `messages`) — not on internal node calls. This makes tests resilient to implementation changes.
- **No mocking library needed** — the graph uses deterministic keyword matching, so plain `unittest` is sufficient. For tests involving real LLMs, use `unittest.mock.patch` to intercept the LLM call.
- **The fallback test (`test_fallback_branch_empty_input`) is the most important** — it proves the graph never panics on unexpected input.

## Step 8 · The `path_map` Pattern and Friendly Labels

Sometimes you want the routing function to return **human-readable labels** (e.g., `"math"`) instead of node names (e.g., `"math_handler"`). The `path_map` argument handles this mapping.

This pattern is useful when:
- Node names are long or implementation-specific
- The routing function is reusable across different graphs with different node names
- You want the routing logic to be readable without knowing the graph structure

In [ ]:
# Alternative routing function that returns friendly labels (not node names)
def route_with_labels(state: RouterState) -> Literal["math", "code", "fallback"]:
    intent = state.get("intent", "unclear")
    if intent == "math":
        return "math"      # friendly label — not a node name
    elif intent == "code":
        return "code"
    else:
        return "fallback"

# Build an equivalent graph using path_map
labeled_builder = StateGraph(RouterState)
labeled_builder.add_node("classify",        classify_intent)
labeled_builder.add_node("math_handler",    math_handler)
labeled_builder.add_node("code_handler",    code_handler)
labeled_builder.add_node("clarify_handler", clarify_handler)

labeled_builder.add_edge(START, "classify")

# path_map maps {label_returned_by_fn: actual_node_name}
labeled_builder.add_conditional_edges(
    "classify",
    route_with_labels,
    {
        "math":     "math_handler",    # "math" label → math_handler node
        "code":     "code_handler",
        "fallback": "clarify_handler",
    }
)

labeled_builder.add_edge("math_handler",    END)
labeled_builder.add_edge("code_handler",    END)
labeled_builder.add_edge("clarify_handler", END)

labeled_graph = labeled_builder.compile()

# Verify it works identically to the original
test_result = labeled_graph.invoke({
    "messages": [HumanMessage(content="calculate 3 * 9")],
    "intent": "",
    "result": "",
})
print("Labeled graph — intent :", test_result["intent"])
print("Labeled graph — result :", test_result["result"][:60])

### What just happened?

- The labeled graph is **functionally identical** to the original — it produces the same output. The only difference is the routing function returns `"math"` instead of `"math_handler"`, and `path_map` translates it.
- **`path_map` is a compile-time contract** — LangGraph validates that every possible return value of the routing function appears as a key in the map. Missing keys cause a `ValueError` at compile time.
- **Use the path_map pattern when** the routing function should be independent of node naming conventions — e.g., a shared router library used across multiple courses or graphs.

In [ ]:
# Challenge: Extend the router with a fourth branch and a confidence score
#
# 1. Add a new intent category: 'explain' — triggered by keywords like
#    'explain', 'what is', 'describe', 'definition', 'how does'
#
# 2. Add a new worker node `explain_handler` that returns:
#    {"result": "[EXPLAIN] Here is a clear explanation: ...",
#     "messages": [AIMessage(content=...)]}
#
# 3. Add a `confidence` field to RouterState (int, 0-100, no reducer needed)
#    Update classify_intent to set confidence:
#    - 90 when a keyword matches
#    - 30 when no keyword matches (fallback)
#
# 4. Update route_by_intent to also check confidence:
#    if confidence < 50 always route to clarify_handler, regardless of intent
#
# 5. Wire the new graph with all four worker branches + the fallback.
#
# 6. Write 3 unit tests:
#    - 'explain what is a neural network' → intent='explain', confidence=90
#    - 'zxqwerty' → confidence=30, routes to clarify
#    - 'calculate the sum' → intent='math', confidence=90

# TODO: extend RouterState with confidence: int
# TODO: implement explain_handler
# TODO: update classify_intent to set confidence
# TODO: update routing to check confidence threshold
# TODO: build and compile extended graph
# TODO: write and run three unit tests

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `add_conditional_edges(src, fn, map)` | Calls `fn(state)` after `src` runs; uses return value to pick next node |
| Routing function contract | Must return a string that **exactly matches** a node name or END |
| `path_map` | Maps friendly labels → node names; validated at compile time |
| Classification node vs routing function | Node writes intent to state; function reads state → picks branch |
| Fallback branch | Always include one — prevents graph hangs on unexpected input |
| Testing approach | Assert on final state dict; test all branches including fallback |
| Stream for debugging | Shows which nodes ran; proves which branch was taken |

> **Tip:** Routing functions must return a string that exactly matches a registered node name or the special END constant. A typo here produces a silent graph hang, not a Python error.

---
## What's next
**Day 4** → Memory, Persistence, and Checkpointers — how to save graph state between invocations, implement conversation memory across sessions, and use LangGraph's built-in checkpointing backends.

Mark Day 3 complete in your [tracker](../index.html).